# Latency and Cost Tradeoff [Step 5 - What Reranking Actually Costs]

> **MLCourse - Agentic AI - Advanced RAG - Reranking**

Notebook 04 gave us a quality number. This notebook gives us the other half of
the decision: **what you pay for it**, in milliseconds and in money.

We measure the real latency of each stage on this machine, chart how cost grows
with the candidate count, compare a local cross-encoder against an LLM-based
reranker on Groq, and finish with a decision checklist.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


In [4]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(p) for p in paragraphs])

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)

print("BM25 documents :", len(paragraphs))
print("dense vectors  :", doc_vectors.shape)


def bm25_rank(query, top_n=50):
    """Return document indices ranked by BM25 score (best first)."""
    scores = bm25.get_scores(tokenize(query))
    return list(np.argsort(scores)[::-1][:top_n])


def dense_rank(query, top_n=50):
    """Return document indices ranked by cosine similarity (best first)."""
    qv = encoder.encode([query], normalize_embeddings=True)[0]
    sims = doc_vectors @ qv
    return list(np.argsort(sims)[::-1][:top_n])


def rrf(rankings, k=60, top_n=50):
    """Reciprocal Rank Fusion - the exact algorithm taught in
    ../01_hybrid_search/03_reciprocal_rank_fusion.ipynb."""
    scores = {}
    for ranked in rankings:
        for rank, doc_id in enumerate(ranked, 1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    ordered = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [int(doc_id) for doc_id, _ in ordered[:top_n]]


def hybrid_rank(query, top_n=50):
    """BM25 + dense, fused with RRF. This is our first-stage retriever."""
    return rrf([bm25_rank(query, top_n), dense_rank(query, top_n)], top_n=top_n)


print("hybrid top-3 for 'the queen and the croquet game':")
for i, doc_id in enumerate(hybrid_rank("the queen and the croquet game", 3), 1):
    print(f"  {i}. doc_{doc_id}: {paragraphs[doc_id][:90]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BM25 documents : 237
dense vectors  : (237, 384)
hybrid top-3 for 'the queen and the croquet game':
  1. doc_150: “Get to your places!” shouted the Queen in a voice of thunder, and people began running ab...
  2. doc_105: The Fish-Footman began by producing from under his arm a great letter, nearly as large as ...
  3. doc_152: The players all played at once without waiting for turns, quarrelling all the while, and f...


### 2. Where the milliseconds go

A two-stage RAG query has four costs. Only one of them grows with the corpus,
and only one grows with the candidate count.

```
   embed query      ~ constant           (one short forward pass)
   vector search    ~ O(log N) with HNSW (grows slowly with corpus size)
   rerank           ~ O(candidates)      (grows with YOUR choice)
   LLM generation   ~ O(context tokens)  (usually dominates everything)
```

Let us measure each one instead of guessing.

In [5]:
from sentence_transformers import CrossEncoder
import numpy as np

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
QUERY = "What game does the Queen of Hearts make everyone play?"
REPEATS = 5


def timed(fn, repeats=REPEATS):
    fn()                                    # warm-up, excluded
    t0 = time.time()
    for _ in range(repeats):
        out = fn()
    return 1000 * (time.time() - t0) / repeats, out


embed_ms, _ = timed(lambda: encoder.encode([QUERY], normalize_embeddings=True))
dense_ms, _ = timed(lambda: dense_rank(QUERY, 50))
bm25_ms, _ = timed(lambda: bm25_rank(QUERY, 50))
hybrid_ms, cands = timed(lambda: hybrid_rank(QUERY, 50))
rerank_ms, _ = timed(lambda: cross_encoder.predict(
    [(QUERY, paragraphs[i]) for i in cands], batch_size=32))

print(f"{'stage':<32}{'ms':>9}")
print("-" * 41)
for label, value in [("embed query", embed_ms), ("dense search (237 docs)", dense_ms),
                     ("bm25 search (237 docs)", bm25_ms),
                     ("hybrid RRF, 50 candidates", hybrid_ms),
                     ("cross-encoder, 50 candidates", rerank_ms)]:
    print(f"{label:<32}{value:>9.1f}")
print("-" * 41)
print(f"{'total retrieval + rerank':<32}{hybrid_ms + rerank_ms:>9.1f}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

stage                                  ms
-----------------------------------------
embed query                           8.6
dense search (237 docs)               7.4
bm25 search (237 docs)                0.3
hybrid RRF, 50 candidates             8.5
cross-encoder, 50 candidates        684.1
-----------------------------------------
total retrieval + rerank            692.6


### 3. The candidate-count curve

This is the knob you actually tune. Reranking cost is linear in candidates,
while quality gains flatten out - so there is a sweet spot, and it is usually
between 30 and 100.

In [6]:
CANDIDATE_COUNTS = [5, 10, 20, 50, 100, 200]

print(f"{'candidates':>11}{'rerank ms':>12}{'ms/candidate':>15}")
print("-" * 38)
curve = []
for n in CANDIDATE_COUNTS:
    ids = hybrid_rank(QUERY, top_n=n)
    ids = (ids * ((n // max(len(ids), 1)) + 1))[:n]   # corpus is small; pad to size n
    ms, _ = timed(lambda ids=ids: cross_encoder.predict(
        [(QUERY, paragraphs[i]) for i in ids], batch_size=32), repeats=3)
    curve.append((n, ms))
    print(f"{n:>11}{ms:>12.1f}{ms / n:>15.2f}")

 candidates   rerank ms   ms/candidate
--------------------------------------


          5        69.7          13.93


         10        93.8           9.38


         20       291.9          14.59


         50       603.0          12.06


        100       694.6           6.95


        200      1522.5           7.61


In [7]:
slope = (curve[-1][1] - curve[0][1]) / (curve[-1][0] - curve[0][0])
print(f"marginal cost: ~{slope:.2f} ms per additional candidate\n")
print("budget planning:")
for budget_ms in [50, 100, 250, 500]:
    affordable = int(budget_ms / slope)
    print(f"  a {budget_ms:>3} ms rerank budget buys ~{affordable:>4} candidates")

marginal cost: ~7.45 ms per additional candidate

budget planning:
  a  50 ms rerank budget buys ~   6 candidates
  a 100 ms rerank budget buys ~  13 candidates
  a 250 ms rerank budget buys ~  33 candidates
  a 500 ms rerank budget buys ~  67 candidates


### 4. Cross-encoder versus LLM-as-reranker

There is a second way to rerank: ask an LLM to order the candidates. It is more
flexible - you can express arbitrary criteria in the prompt ("prefer recent
documents", "prefer official sources") - but it is far more expensive per query
and it consumes tokens from the same rate limit your generation step needs.

Let us actually run one and time it against Groq.

In [8]:
candidates = hybrid_rank(QUERY, top_n=10)

listing = "\n".join(f"[{i}] {paragraphs[doc_id][:220]}"
                    for i, doc_id in enumerate(candidates))

t0 = time.time()
llm_order = ask(
    "You are a search reranker. Below are 10 candidate passages. "
    "Return ONLY the indices of the 5 most relevant passages for the query, "
    "best first, as a comma-separated list of numbers and nothing else.\n\n"
    f"Query: {QUERY}\n\nPassages:\n{listing}"
)
llm_rerank_ms = 1000 * (time.time() - t0)

print("raw model output:", llm_order)

picked = [int(x) for x in re.findall(r"\d+", llm_order) if int(x) < len(candidates)][:5]
llm_ids = [candidates[i] for i in picked]

ce_scores = cross_encoder.predict([(QUERY, paragraphs[i]) for i in candidates])
ce_ids = [int(i) for i, _ in sorted(zip(candidates, ce_scores), key=lambda x: -x[1])][:5]

print("\nLLM reranker top 5    :", llm_ids)
print("cross-encoder top 5   :", ce_ids)
print("overlap               :", len(set(llm_ids) & set(ce_ids)), "of 5")

raw model output: 4, 2, 5, 8, 3

LLM reranker top 5    : [105, 150, 152, 173, 157]
cross-encoder top 5   : [152, 200, 173, 150, 172]
overlap               : 3 of 5


In [9]:
ce_10_ms, _ = timed(lambda: cross_encoder.predict(
    [(QUERY, paragraphs[i]) for i in candidates], batch_size=32), repeats=3)

print(f"{'reranker':<26}{'ms (10 candidates)':>20}{'tokens':>10}")
print("-" * 56)
print(f"{'cross-encoder (local)':<26}{ce_10_ms:>20.1f}{'0':>10}")
print(f"{'LLM via Groq':<26}{llm_rerank_ms:>20.1f}{'~1500':>10}")
print(f"\nthe cross-encoder is ~{llm_rerank_ms / ce_10_ms:.0f}x faster here and "
      f"consumes none of your 8000 tokens/minute budget")

reranker                    ms (10 candidates)    tokens
--------------------------------------------------------
cross-encoder (local)                    108.0         0
LLM via Groq                            1711.7     ~1500

the cross-encoder is ~16x faster here and consumes none of your 8000 tokens/minute budget


### When each one is the right call

**Cross-encoder** - the default. Local, no API dependency, no token cost, tens
of milliseconds, scales to hundreds of candidates. Its judgement is fixed: pure
query-document relevance.

**LLM reranker** - reach for it when relevance depends on criteria a MS MARCO
model was never trained on: recency, source authority, user role, regulatory
scope, "prefer the document that contradicts the claim". Keep the candidate list
short (10-20), and remember you are spending the same rate limit as generation.

**Hosted rerank APIs** (Cohere Rerank, Voyage) sit between the two: better
quality than a local MiniLM, priced per search unit, and they add a network
round-trip to every query.

### 5. Putting rerank latency in perspective

The honest framing: compare the reranker's cost to the cost of the LLM call it
is feeding.

In [10]:
context = "\n\n".join(f"[doc_{i}] {paragraphs[i]}" for i in ce_ids[:3])
prompt = ("Answer using ONLY the context.\n\n" + context +
          f"\n\nQuestion: {QUERY}\nAnswer:")

t0 = time.time()
answer = ask(prompt)
gen_ms = 1000 * (time.time() - t0)

retrieval_ms = hybrid_ms + rerank_ms
total = retrieval_ms + gen_ms

print("answer:", answer[:300])
print()
print(f"{'component':<34}{'ms':>9}{'share':>9}")
print("-" * 52)
print(f"{'retrieval + rerank':<34}{retrieval_ms:>9.0f}{retrieval_ms / total * 100:>8.1f}%")
print(f"{'LLM generation (Groq)':<34}{gen_ms:>9.0f}{gen_ms / total * 100:>8.1f}%")
print("-" * 52)
print(f"{'end to end':<34}{total:>9.0f}{100.0:>8.1f}%")

answer: Based on the provided context, the text does not explicitly name the specific game the Queen of Hearts makes everyone play. It only describes the players "playing at once without waiting for turns," "quarrelling," "fighting for the hedgehogs," and the Queen shouting "Off with his head!" or "Off with

component                                ms    share
----------------------------------------------------
retrieval + rerank                      693    27.8%
LLM generation (Groq)                  1797    72.2%
----------------------------------------------------
end to end                             2490   100.0%


For most systems the generation call dominates, and the reranker is a small
slice of end-to-end latency. That is why reranking is such a common production
choice: it buys a real quality improvement out of a budget the user cannot
perceive.

The exception is autocomplete-style or per-keystroke retrieval, where the whole
budget is 50 ms and there is no LLM call at all. There, a reranker is a
significant fraction of the response and the tradeoff genuinely changes.

### 6. A decision checklist

Ship the reranker when:

- [ ] Measured precision@k or MRR improves on **your** evaluation set
      (notebook 04, not someone else's benchmark).
- [ ] Rerank latency is a small share of end-to-end latency.
- [ ] Your corpus is large or noisy enough that stage 1 returns real distractors.
- [ ] You can widen stage 1 to 30+ candidates - otherwise there is nothing to
      rerank.

Skip it when:

- [ ] The corpus is small and clean and stage 1 is already near-perfect.
- [ ] You are latency-bound with no LLM call to hide behind.
- [ ] Your bottleneck is recall, not precision - fix chunking, retrieval, or
      query transformation first
      (see [`../12_query_transformation`](../12_query_transformation/README.md)).

### 7. Key takeaways

- Reranking cost is **linear in candidate count** and independent of corpus
  size; the candidate count is the knob.
- Measure the marginal ms-per-candidate on your hardware and buy candidates with
  an explicit latency budget.
- A local cross-encoder is roughly two orders of magnitude cheaper than an LLM
  reranker and uses none of your token budget.
- Rerank latency is usually small next to generation - which is why the pattern
  is worth it in most, but not all, systems.

That closes the reranking module. Next module:
[`../12_query_transformation`](../12_query_transformation/README.md), which
improves the *query* rather than the ranking.